In [1]:
import sys
PATH_TO_PREDICTOR_HOME = ".."
sys.path.append(PATH_TO_PREDICTOR_HOME)


from mhcnames import normalize_allele_name
import numpy as np
import pandas as pd
from config import RESULT_COLUMNS


## Read/Make the data

In [2]:
ALLELES = ['H-2Ld', 'H-2IEd']


per_allele_wt_seq = {
    "H-2Ld": "LDLSANCLPSLPSIIPWGLINLKKL",
    "H-2IEd": "GRPDNTGRGYVLRRILRRAVRYSHE"
}
per_allele_mt_seq = {
    "H-2Ld": "LDLSANCLPSLPFIIPWGLINLKKL",
     "H-2IEd": "GRPDNTGRGYVLIRILRRAVRYSHE"
}

per_allele_mutation_position = {
    "H-2Ld": 12,
    "H-2IEd": 12
}

per_class_alleles = {
    "I": {"H-2Ld"},
    "II": {"H-2IEd"}
}

per_class_min_length = {
    "I": 8,
    "II": 9
}

per_class_max_length = {
    "I": 12,
    "II": 25
}



def defineClass(allele_name):
    for class_to_use in per_class_alleles:
        if allele_name in per_class_alleles[class_to_use]:
            return class_to_use
    print("Allele Not found")
            
per_allele_mt_peptides = {allele_name : list() for allele_name in ALLELES}
per_allele_wt_peptides= {allele_name : list() for allele_name in ALLELES}
per_allele_total_df = dict()
per_allele_mt_df = dict()
per_allele_wt_df = dict()

for allele_name in ALLELES:
    wt_seq = per_allele_wt_seq[allele_name]
    mt_seq = per_allele_mt_seq[allele_name]
    total_length_mt = len(mt_seq)
    class_to_use = defineClass(allele_name)
    min_length = per_class_min_length[class_to_use]
    max_length = per_class_max_length[class_to_use]
    
    mutation_position = per_allele_mutation_position[allele_name]
    
    for target_length in range(min_length, max_length + 1):
        start_index = mutation_position - target_length  + 1 if mutation_position - target_length  > 0 else 0
        end_index = mutation_position
        for i in range(start_index, end_index + 1):
            #mt
            candidate_str = mt_seq[i: i + target_length]
            if len(candidate_str) == target_length:
                per_allele_mt_peptides[allele_name].append(candidate_str)
            #wt
            candidate_str = wt_seq[i: i + target_length]
            if len(candidate_str) == target_length:
                per_allele_wt_peptides[allele_name].append(candidate_str)    
            
#                 print(len(candidate_str), candidate_str)
    df_mt = pd.DataFrame({"peptide": per_allele_mt_peptides[allele_name]})
    df_mt['length'] = df_mt['peptide'].str.len()
    df_mt['allele'] = allele_name
    df_mt[RESULT_COLUMNS['id']] =  'seq' + df_mt.index.astype(str).values
    
    df_wt = pd.DataFrame({"peptide": per_allele_wt_peptides[allele_name]})
    df_wt['length'] = df_wt['peptide'].str.len()
    df_wt['allele'] = allele_name
    df_wt[RESULT_COLUMNS['id']] =  'seq' + df_wt.index.astype(str).values

    
    per_allele_mt_df[allele_name] = df_mt
    per_allele_wt_df[allele_name] = df_wt
    

    

    
    df_mt = df_mt.rename(columns={"peptide": "peptide_mt"})
    
    df_mt["peptide_wt"] = df_wt['peptide']
    per_allele_total_df[allele_name] = df_mt
    
#stack all the dfs
list_of_df = per_allele_total_df.values()
total_df = pd.concat(list_of_df)



In [3]:
total_df

,peptide_mt,length,allele,source_sequence_name,peptide_wt
0,NCLPSLPF,8,H-2Ld,seq0,NCLPSLPS
1,CLPSLPFI,8,H-2Ld,seq1,CLPSLPSI
2,LPSLPFII,8,H-2Ld,seq2,LPSLPSII
3,PSLPFIIP,8,H-2Ld,seq3,PSLPSIIP
4,SLPFIIPW,8,H-2Ld,seq4,SLPSIIPW
...,...,...,...,...,...
129,RPDNTGRGYVLIRILRRAVRYSH,23,H-2IEd,seq129,RPDNTGRGYVLRRILRRAVRYSH
130,PDNTGRGYVLIRILRRAVRYSHE,23,H-2IEd,seq130,PDNTGRGYVLRRILRRAVRYSHE
131,GRPDNTGRGYVLIRILRRAVRYSH,24,H-2IEd,seq131,GRPDNTGRGYVLRRILRRAVRYSH
132,RPDNTGRGYVLIRILRRAVRYSHE,24,H-2IEd,seq132,RPDNTGRGYVLRRILRRAVRYSHE


# Write to files

In [19]:
DATA_PATH = "/home/denklewer/Data/schreiber/"


In [20]:
import os
if not os.path.exists(DATA_PATH):
    os.makedirs(DATA_PATH)

In [21]:
# for decoy_key in DECOY_PREFIXES:
for allele_name in ALLELES:
    df_mt = per_allele_mt_df[allele_name]
    df_wt = per_allele_wt_df[allele_name]
    allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    df_mt.to_csv("{}schreiber_data_{}_mt.csv".format(DATA_PATH,allele_name),sep=";", index=False)
    df_wt.to_csv("{}schreiber_data_{}_wt.csv".format(DATA_PATH,allele_name),sep=";", index=False)
    total_df.to_csv("{}schreiber_total_data.csv".format(DATA_PATH),sep=";", index=False)

In [22]:
total_df

,peptide_mt,length,allele,source_sequence_name,peptide_wt
0,NCLPSLPF,8,H-2Ld,seq0,NCLPSLPS
1,CLPSLPFI,8,H-2Ld,seq1,CLPSLPSI
2,LPSLPFII,8,H-2Ld,seq2,LPSLPSII
3,PSLPFIIP,8,H-2Ld,seq3,PSLPSIIP
4,SLPFIIPW,8,H-2Ld,seq4,SLPSIIPW
...,...,...,...,...,...
129,RPDNTGRGYVLIRILRRAVRYSH,23,H-2IEd,seq129,RPDNTGRGYVLRRILRRAVRYSH
130,PDNTGRGYVLIRILRRAVRYSHE,23,H-2IEd,seq130,PDNTGRGYVLRRILRRAVRYSHE
131,GRPDNTGRGYVLIRILRRAVRYSH,24,H-2IEd,seq131,GRPDNTGRGYVLRRILRRAVRYSH
132,RPDNTGRGYVLIRILRRAVRYSHE,24,H-2IEd,seq132,RPDNTGRGYVLRRILRRAVRYSHE
